# 29. ACID & Transactions

*Module: 7 — Advanced server · Companion summary: `lectures/29-acid-transactions.md` · SQL engine: SQLite (in-memory)*

## What you will study

- state what ACID stands for, property by property
- run a COMMIT path and a ROLLBACK path in sqlite
- see isolation in action: the money-transfer example

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 📗 ACID

| Property | Meaning | Transfer example |
|---|---|---|
| **A**tomicity | all statements commit or none | send $200 A→B: both sides, or none |
| **C**onsistency | constraints hold after commit | money never appears from nowhere |
| **I**solation | concurrent transactions don't see each other's mid-states | your balance doesn't show partial deduct |
| **D**urability | committed data survives crash | confirmed transfer stays after power outage |

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 The transfer story — setup

In [ ]:
q("CREATE TABLE accounts (id TEXT PRIMARY KEY, owner TEXT, balance REAL)")
q("INSERT INTO accounts VALUES ('A','Alice',1000), ('B','Bob',300)")
conn.commit()                        # seed data committed, so the demos below
                                     # can open their own explicit transactions
                                     # (sqlite3 otherwise keeps an implicit txn open
                                     #  after writes, and BEGIN inside it fails)
q("SELECT * FROM accounts")

## 🧪 Path 1 — a clean transfer, ends in COMMIT

In [ ]:
amount = 200
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - ? WHERE id = 'A'", (amount,))
    conn.execute("UPDATE accounts SET balance = balance + ? WHERE id = 'B'", (amount,))
    conn.execute("COMMIT")                      # atomicity: both or nothing
    print('✅ transfer committed')
except Exception as e:
    conn.execute("ROLLBACK")
    print('rolled back:', e)
q("SELECT * FROM accounts")

## 🧪 Path 2 — a dirty transfer: rollback undoes everything

Alice has 800 left after the first transfer! Transferring 800 must fail *halfway* —
and rollback must erase the first UPDATE:

In [ ]:
amount = 800
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE accounts SET balance = balance - ? WHERE id = 'A'", (amount,))
    q("SELECT id, balance FROM accounts WHERE id = 'A'")      # visible inside the txn!
    balance = conn.execute("SELECT balance FROM accounts WHERE id = 'A'").fetchone()[0]
    if balance < 300:                                          # keep a minimum balance
        raise ValueError(f'A would drop below minimum (balance={balance})')
    conn.execute("UPDATE accounts SET balance = balance + ? WHERE id = 'B'", (amount,))
    conn.execute("COMMIT")
except ValueError as e:
    conn.execute("ROLLBACK")
    print('❌ transaction rolled back:', e)
q("SELECT * FROM accounts")

## 🧪 Isolation check — a second connection can't see the uncommitted work

We open a **file-based** DB so two connections (= two "sessions") can point at the
exact same database, and watch what each one sees mid-transaction:

In [ ]:
# Use a real file so both connections point at the same DB.
import os, tempfile
p = os.path.join(tempfile.mkdtemp(), 'bank.db')
c1 = sqlite3.connect(p, isolation_level=None)   # autocommit off, manual BEGIN
c2 = sqlite3.connect(p, isolation_level=None)
c1.execute("CREATE TABLE t (v INTEGER)")
c1.execute("INSERT INTO t VALUES (1)")
c1.execute("BEGIN")
c1.execute("INSERT INTO t VALUES (2)")
print('c1 sees:', c1.execute('SELECT * FROM t').fetchall())          # 1,2
print('c2 sees:', c2.execute('SELECT * FROM t').fetchall())          # only [1]!
c1.execute("COMMIT")
print('after commit, c2 sees:', c2.execute('SELECT * FROM t').fetchall())
db_path = p   # (kept for reference)

## 💡 Tips & tricks

- Golden habits from the course: keep transactions **short**, commit at a
  *logical point*, always pair every error path with ROLLBACK.
- Python's `with connection:` is a one-line transaction — nb22 showed both spellings.
- Durability in SQLite: a `journal_mode=WAL` file db survives crashes mid-write.

## 🎯 Your turn

1. Transfer 100 from B to A twice in one transaction — commit, check balances add up.
2. Make a transaction fail *by constraint* (transfer more than A's balance into a
   `CHECK (balance >= 0)`-guarded table... try the CHECK!) and watch rollback fire.

In [ ]:
# 💡 Solution
q("BEGIN")
conn.execute("UPDATE accounts SET balance = balance - 100 WHERE id = 'B'")
conn.execute("UPDATE accounts SET balance = balance + 100 WHERE id = 'A'")
conn.execute("UPDATE accounts SET balance = balance - 100 WHERE id = 'B'")
conn.execute("UPDATE accounts SET balance = balance + 100 WHERE id = 'A'")
conn.execute("COMMIT")
q("SELECT * FROM accounts")

c = sqlite3.connect(os.path.join(tempfile.mkdtemp(), 'guarded.db'), isolation_level=None)
c.execute("CREATE TABLE acct (id TEXT PRIMARY KEY, balance REAL CHECK (balance >= 0))")
c.execute("INSERT INTO acct VALUES ('X', 100)")
c.execute("BEGIN")
try:
    c.execute("UPDATE acct SET balance = -50 WHERE id = 'X'")
    print('constraint unexpectedly allowed it!')
except sqlite3.IntegrityError as e:
    print('constraint fired:', e)
c.execute("ROLLBACK")
print('after constraint rollback:', c.execute('SELECT * FROM acct').fetchall())